In [36]:
import pandas as pd 
import numpy as np
from sklearn.preprocessing import LabelEncoder , OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix , accuracy_score
from sklearn.pipeline import Pipeline


In [2]:
data = pd.read_csv('titanic.csv')
data.head(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
data.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')

In [4]:
data['Age'].isnull().sum()
data['Age'] = data['Age'].fillna(data['Age'].mean())
data['Age'].isnull().sum()

np.int64(0)

In [5]:
data["Fare"] = data["Fare"].fillna(data["Fare"].mean())

In [6]:
data['Embarked'].isnull().sum()

np.int64(2)

In [7]:
data = data.drop(columns=['Cabin'])

In [8]:
data = data.drop(columns=['Ticket','Name'])

In [9]:
data = data.drop(columns=['PassengerId'])

In [10]:
data.head(5)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [11]:
def age_group(age):
    if age < 18 :
        return "Child"
    elif age > 18 :
        return "Adult"
    else :
        return "Old"
data['Age_Group'] = data['Age'].apply(age_group)

data['Family'] = data['SibSp'] + data['Parch']


In [12]:
# Encoding 
le = LabelEncoder()
data['Sex'] = le.fit_transform(data['Sex'])
data['Age_Group'] = le.fit_transform(data['Age_Group'])

In [13]:
one = OneHotEncoder(sparse_output=False)

embarked_encoded = one.fit_transform(data[['Embarked']])

embarked_df = pd.DataFrame(embarked_encoded, columns=one.get_feature_names_out())

data = pd.concat([data, embarked_df], axis=1)

data.drop('Embarked', axis=1, inplace=True)

In [14]:
data.head(5)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Age_Group,Family,Embarked_C,Embarked_Q,Embarked_S,Embarked_nan
0,0,3,1,22.0,1,0,7.2500,0,1,0.0,0.0,1.0,0.0
1,1,1,0,38.0,1,0,71.2833,0,1,1.0,0.0,0.0,0.0
2,1,3,0,26.0,0,0,7.9250,0,0,0.0,0.0,1.0,0.0
3,1,1,0,35.0,1,0,53.1000,0,1,0.0,0.0,1.0,0.0
4,0,3,1,35.0,0,0,8.0500,0,0,0.0,0.0,1.0,0.0


In [15]:
sc = StandardScaler()
data[['Age','Fare']] = sc.fit_transform(data[['Age','Fare']])

In [16]:
model_LR = LogisticRegression()
X  = data.drop('Survived',axis=1)
Y = data['Survived']

In [17]:
X_train, X_test , Y_train , Y_test = train_test_split(X, Y, test_size=0.2, random_state=0) 

In [18]:
model_LR.fit(X_train,Y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [19]:
X_train.isnull().sum()

Pclass          0
Sex             0
Age             0
SibSp           0
Parch           0
Fare            0
Age_Group       0
Family          0
Embarked_C      0
Embarked_Q      0
Embarked_S      0
Embarked_nan    0
dtype: int64

In [20]:
y_pred_LR = model_LR.predict(X_test)

In [21]:
# accuracy Check 
cm = confusion_matrix(Y_test,y_pred_LR)
print("cm:",cm)
ac = accuracy_score(Y_test,y_pred_LR)
print("accuracy:",ac)

cm: [[92 18]
 [18 51]]
accuracy: 0.7988826815642458


In [22]:
model_knn = KNeighborsClassifier()
model_knn.fit(X_train,Y_train)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [23]:
y_pred_knn = model_knn.predict(X_test)

In [24]:
# Accuracy Check 
cm = confusion_matrix (Y_test,y_pred_knn)
print("cm:",cm,)
ac = accuracy_score(Y_test,y_pred_knn)
print("ac:",ac)

cm: [[94 16]
 [20 49]]
ac: 0.7988826815642458


In [25]:
#prediction 
result = pd.DataFrame({
    'Actual Servive' : Y_test,
    'predicted Servive' : y_pred_knn
    })
result.head(5)

,Actual Servive,predicted Servive
495,0,0
648,0,0
278,0,0
31,1,1
255,1,1


In [26]:
model_rfc = RandomForestClassifier()
model_rfc.fit(X_train,Y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [27]:
y_pred_rfc = model_rfc.predict(X_test)

In [28]:
# Accuracy check
cm = confusion_matrix(Y_test,y_pred_rfc)
print("CM : " ,cm)
ac = accuracy_score(Y_test,y_pred_rfc)
print('AC : ',ac)

CM :  [[99 11]
 [19 50]]
AC :  0.8324022346368715


In [37]:
model_xg = XGBClassifier()
model_xg.fit(X_train,Y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [38]:
y_pred_xg = model_xg.predict(X_test)

In [40]:
# Accuracy Check 
cm = confusion_matrix(Y_test,y_pred_xg)
print('cm:',cm)
ac = accuracy_score(Y_test,y_pred_xg)
print('ac:',ac)

cm: [[101   9]
 [ 18  51]]
ac: 0.8491620111731844
